## AutoClean: Automated Data Cleaning Framework

### Objective
Build a reusable data-cleaning pipeline that can work with any structured dataset by applying data-type-based cleaning rules.

### Dataset
US Accidents Dataset (Used for validation)

### Cleaning Operations
- Column Standardization
- Missing Value Treatment
- Duplicate Removal
- Text Cleaning
- Outlier Handling
- Data Quality Reporting

## Step 1: Importing Libraries

In [1]:
# ==========================================================
# STEP 1: IMPORT REQUIRED LIBRARIES
# ==========================================================
#
# Purpose:
# Import all libraries required for data cleaning
#
# ==========================================================

import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

print("=" * 60)
print("LIBRARIES IMPORTED SUCCESSFULLY")
print("=" * 60)

LIBRARIES IMPORTED SUCCESSFULLY


## Step 2: Load Dataset

In [ ]:
# ==========================================================
# STEP 2: LOAD DATASET
# ==========================================================
#
# Purpose:
# Load raw dataset into a Pandas DataFrame
#
# ==========================================================

file_path = "../data/US_Accidents_March23.csv"

df = pd.read_csv(file_path, encoding= 'latin1')

print("=" * 60)
print("DATASET LOADED SUCCESSFULLY")
print("=" * 60)

print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

DATASET LOADED SUCCESSFULLY
Rows    : 7,728,394
Columns : 46


## Step 3: Initial Dataset Profiling

In [3]:
# ==========================================================
# STEP 3: INITIAL DATASET PROFILING
# ==========================================================
#
# Purpose:
# Understand the structure and quality of the dataset
# before applying cleaning operations.
#
# Metrics:
# - Rows
# - Columns
# - Data Types
# - Missing Values
# - Duplicate Records
# - Memory Usage
#
# ==========================================================

numeric_cols = df.select_dtypes(include=np.number).columns

categorical_cols = df.select_dtypes(
    include=["object", "string"]
).columns

datetime_cols = df.select_dtypes(
    include=["datetime64[ns]"]
).columns

boolean_cols = df.select_dtypes(
    include="bool"
).columns

total_missing = df.isnull().sum().sum()

duplicate_records = df.duplicated().sum()

memory_usage_mb = (
    df.memory_usage(deep=True).sum()
    / 1024**2
)

print("=" * 60)
print("INITIAL DATASET PROFILE")
print("=" * 60)

print(f"Rows                : {len(df):,}")
print(f"Columns             : {len(df.columns)}")

print("\nCOLUMN TYPE SUMMARY")
print("-" * 60)

print(f"Numeric Columns     : {len(numeric_cols)}")
print(f"Categorical Columns : {len(categorical_cols)}")
print(f"Datetime Columns    : {len(datetime_cols)}")
print(f"Boolean Columns     : {len(boolean_cols)}")

print("\nDATA QUALITY SUMMARY")
print("-" * 60)

print(f"Missing Values      : {total_missing:,}")
print(f"Duplicate Records   : {duplicate_records:,}")

print("\nMEMORY USAGE")
print("-" * 60)

print(f"Dataset Size (MB)   : {memory_usage_mb:.2f}")

print("\nTOP 10 COLUMNS WITH MOST MISSING VALUES")
print("-" * 60)

missing_summary = (
    df.isnull()
      .sum()
      .sort_values(ascending=False)
      .head(10)
)

print(missing_summary)

print("=" * 60)

INITIAL DATASET PROFILE
Rows                : 7,728,394
Columns             : 46

COLUMN TYPE SUMMARY
------------------------------------------------------------
Numeric Columns     : 13
Categorical Columns : 20
Datetime Columns    : 0
Boolean Columns     : 13

DATA QUALITY SUMMARY
------------------------------------------------------------
Missing Values      : 12,840,498
Duplicate Records   : 0

MEMORY USAGE
------------------------------------------------------------
Dataset Size (MB)   : 9695.59

TOP 10 COLUMNS WITH MOST MISSING VALUES
------------------------------------------------------------
End_Lat              3402762
End_Lng              3402762
Precipitation(in)    2203586
Wind_Chill(F)        1999019
Wind_Speed(mph)       571233
Visibility(mi)        177098
Wind_Direction        175206
Humidity(%)           174144
Weather_Condition     173459
Temperature(F)        163853
dtype: int64


## Step 4: Store Baseline Metrics

In [4]:
# ==========================================================
# STEP 4: STORE BASELINE METRICS
# ==========================================================
#
# Purpose:
# Save dataset statistics before cleaning.
# These metrics will be used in the final report.
#
# ==========================================================

baseline_rows = len(df)

baseline_columns = len(df.columns)

baseline_missing = df.isnull().sum().sum()

baseline_duplicates = df.duplicated().sum()

baseline_memory = (
    df.memory_usage(deep=True).sum()
    / 1024**2
)

print("=" * 60)
print("BASELINE METRICS STORED")
print("=" * 60)

print(f"Rows              : {baseline_rows:,}")
print(f"Columns           : {baseline_columns}")
print(f"Missing Values    : {baseline_missing:,}")
print(f"Duplicate Records : {baseline_duplicates:,}")
print(f"Memory Usage (MB) : {baseline_memory:.2f}")

BASELINE METRICS STORED
Rows              : 7,728,394
Columns           : 46
Missing Values    : 12,840,498
Duplicate Records : 0
Memory Usage (MB) : 9695.59


## Step  5: Standardize Column Names

In [5]:
# ==========================================================
# STEP 5: STANDARDIZE COLUMN NAMES
# ==========================================================
#
# Purpose:
# Create consistent column naming convention.
#
# Rules:
# - Remove leading/trailing spaces
# - Convert to lowercase
# - Replace spaces with underscores
#
# ==========================================================

old_columns = df.columns.tolist()

df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_", regex=False)
)

new_columns = df.columns.tolist()

print("=" * 60)
print("COLUMN STANDARDIZATION REPORT")
print("=" * 60)

print(f"Columns Processed : {len(df.columns)}")

print("\nSample Changes")
print("-" * 60)

for old, new in zip(old_columns[:10], new_columns[:10]):
    print(f"{old}  -->  {new}")

print("=" * 60)

COLUMN STANDARDIZATION REPORT
Columns Processed : 46

Sample Changes
------------------------------------------------------------
ID  -->  id
Source  -->  source
Severity  -->  severity
Start_Time  -->  start_time
End_Time  -->  end_time
Start_Lat  -->  start_lat
Start_Lng  -->  start_lng
End_Lat  -->  end_lat
End_Lng  -->  end_lng
Distance(mi)  -->  distance(mi)


## Step 6: Date Format Detection & Conversion

In [6]:
# ==========================================================
# STEP 6: DATE FORMAT DETECTION & CONVERSION
# ==========================================================
#
# Purpose:
# Automatically identify date-like columns and convert
# them into datetime format.
#
# Logic:
# Attempt conversion on object columns.
# If at least 80% of values convert successfully,
# treat the column as a datetime column.
#
# ==========================================================

date_columns_converted = []

object_columns = df.select_dtypes(
    include=["object", "string"]
).columns

for col in object_columns:

    try:

        # Attempt datetime conversion
        converted = pd.to_datetime(
            df[col],
            errors="coerce"
        )

        # Calculate successful conversion rate
        conversion_rate = (
            converted.notna().sum()
            / len(df)
        )

        # If most values convert successfully,
        # classify column as datetime

        if conversion_rate >= 0.80:

            df[col] = converted

            date_columns_converted.append(col)

    except:
        pass

print("=" * 60)
print("DATE FORMAT CONVERSION REPORT")
print("=" * 60)

print(f"Object Columns Checked : {len(object_columns)}")
print(f"Date Columns Converted : {len(date_columns_converted)}")

if len(date_columns_converted) > 0:

    print("\nConverted Columns")
    print("-" * 60)

    for col in date_columns_converted:
        print(f"✓ {col}")

else:

    print("\nNo date columns detected.")

print("=" * 60)

DATE FORMAT CONVERSION REPORT
Object Columns Checked : 20
Date Columns Converted : 3

Converted Columns
------------------------------------------------------------
✓ start_time
✓ end_time
✓ weather_timestamp


## Step 7: Post-Date Conversion Profiling

In [7]:
# ==========================================================
# STEP 7: POST-DATE CONVERSION PROFILE
# ==========================================================
#
# Purpose:
# Re-profile dataset after datetime conversion.
#
# ==========================================================

numeric_cols = df.select_dtypes(include=np.number).columns

categorical_cols = df.select_dtypes(
    include=["object", "string"]
).columns

datetime_cols = df.select_dtypes(
    include=["datetime64[ns]"]
).columns

boolean_cols = df.select_dtypes(
    include="bool"
).columns

print("=" * 60)
print("POST-DATE CONVERSION PROFILE")
print("=" * 60)

print(f"Numeric Columns     : {len(numeric_cols)}")
print(f"Categorical Columns : {len(categorical_cols)}")
print(f"Datetime Columns    : {len(datetime_cols)}")
print(f"Boolean Columns     : {len(boolean_cols)}")

print("=" * 60)

POST-DATE CONVERSION PROFILE
Numeric Columns     : 13
Categorical Columns : 17
Datetime Columns    : 3
Boolean Columns     : 13


## Step 8: Handle Missing Values

In [8]:
# ==========================================================
# STEP 8: HANDLE MISSING VALUES
# ==========================================================
#
# Purpose:
# Handle missing values using data-type-based rules.
#
# Rules:
# Numeric:
#   - Mean if approximately symmetric
#   - Median if skewed
#
# Categorical:
#   - Mode
#
# Datetime:
#   - Keep Null values
#
# ==========================================================

total_filled = 0

numeric_processed = 0
categorical_processed = 0

numeric_fill_log = []
categorical_fill_log = []

# ------------------------------
# NUMERIC COLUMNS
# ------------------------------

numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:

    missing_before = df[col].isnull().sum()

    if missing_before > 0:

        skewness = df[col].skew()

        if abs(skewness) < 0.5:

            fill_value = df[col].mean()
            method = "Mean"

        else:

            fill_value = df[col].median()
            method = "Median"

        df[col] = df[col].fillna(fill_value)

        total_filled += missing_before
        numeric_processed += 1

        numeric_fill_log.append(
            [col, method, missing_before]
        )

# ------------------------------
# CATEGORICAL COLUMNS
# ------------------------------

categorical_cols = df.select_dtypes(
    include=["object", "string"]
).columns

for col in categorical_cols:

    missing_before = df[col].isnull().sum()

    if missing_before > 0:

        mode_value = df[col].mode(dropna=True)

        if len(mode_value) > 0:

            df[col] = df[col].fillna(mode_value[0])

            total_filled += missing_before
            categorical_processed += 1

            categorical_fill_log.append(
                [col, "Mode", missing_before]
            )

remaining_missing = df.isnull().sum().sum()

print("=" * 60)
print("MISSING VALUE TREATMENT REPORT")
print("=" * 60)

print(f"Numeric Columns Processed     : {numeric_processed}")
print(f"Categorical Columns Processed : {categorical_processed}")

print(f"\nTotal Values Filled          : {total_filled:,}")
print(f"Remaining Missing Values     : {remaining_missing:,}")

print("=" * 60)

MISSING VALUE TREATMENT REPORT
Numeric Columns Processed     : 9
Categorical Columns Processed : 12

Total Values Filled          : 12,720,270
Remaining Missing Values     : 1,606,560


## Step 9: Remove Duplicate Records

In [9]:
# ==========================================================
# STEP 9: REMOVE DUPLICATE RECORDS
# ==========================================================
#
# Purpose:
# Remove duplicate rows from the dataset.
#
# ==========================================================

rows_before = len(df)

df = df.drop_duplicates()

rows_after = len(df)

duplicates_removed = rows_before - rows_after

print("=" * 60)
print("DUPLICATE REMOVAL REPORT")
print("=" * 60)

print(f"Rows Before         : {rows_before:,}")
print(f"Rows After          : {rows_after:,}")
print(f"Duplicates Removed  : {duplicates_removed:,}")

print("=" * 60)

DUPLICATE REMOVAL REPORT
Rows Before         : 7,728,394
Rows After          : 7,728,394
Duplicates Removed  : 0


## Step 10: Clean Text Columns

In [10]:
# ==========================================================
# STEP 10: CLEAN TEXT COLUMNS
# ==========================================================
#
# Purpose:
# Standardize text fields.
#
# ==========================================================

text_columns = df.select_dtypes(
    include=["object", "string"]
).columns

for col in text_columns:

    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.title()
    )

print("=" * 60)
print("TEXT CLEANING REPORT")
print("=" * 60)

print(f"Text Columns Processed : {len(text_columns)}")

print("=" * 60)

TEXT CLEANING REPORT
Text Columns Processed : 17


## Step 11: Handle Outliers (IQR Method)

In [11]:
# ==========================================================
# STEP 11: HANDLE OUTLIERS USING IQR
# ==========================================================
#
# Purpose:
# Detect and cap outliers in numeric columns.
#
# ==========================================================

numeric_cols = df.select_dtypes(
    include=np.number
).columns

total_outliers = 0

for col in numeric_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - (1.5 * IQR)
    upper_bound = Q3 + (1.5 * IQR)

    outliers = (
        (df[col] < lower_bound)
        |
        (df[col] > upper_bound)
    ).sum()

    total_outliers += outliers

    df[col] = np.where(
        df[col] < lower_bound,
        lower_bound,
        df[col]
    )

    df[col] = np.where(
        df[col] > upper_bound,
        upper_bound,
        df[col]
    )

print("=" * 60)
print("OUTLIER HANDLING REPORT")
print("=" * 60)

print(f"Numeric Columns Processed : {len(numeric_cols)}")
print(f"Outliers Capped          : {total_outliers:,}")

print("=" * 60)

OUTLIER HANDLING REPORT
Numeric Columns Processed : 13
Outliers Capped          : 8,826,748


## Step 12: Final Dataset Profiling

In [12]:
# ==========================================================
# STEP 12: FINAL DATASET PROFILING
# ==========================================================
#
# Purpose:
# Profile dataset after all cleaning operations.
#
# ==========================================================

numeric_cols = df.select_dtypes(include=np.number).columns

categorical_cols = df.select_dtypes(
    include=["object", "string"]
).columns

datetime_cols = df.select_dtypes(
    include=["datetime64[ns]"]
).columns

boolean_cols = df.select_dtypes(
    include="bool"
).columns

final_missing = df.isnull().sum().sum()

final_duplicates = df.duplicated().sum()

final_memory = (
    df.memory_usage(deep=True).sum()
    / 1024**2
)

print("=" * 60)
print("FINAL DATASET PROFILE")
print("=" * 60)

print(f"Rows                : {len(df):,}")
print(f"Columns             : {len(df.columns)}")

print("\nCOLUMN TYPE SUMMARY")
print("-" * 60)

print(f"Numeric Columns     : {len(numeric_cols)}")
print(f"Categorical Columns : {len(categorical_cols)}")
print(f"Datetime Columns    : {len(datetime_cols)}")
print(f"Boolean Columns     : {len(boolean_cols)}")

print("\nDATA QUALITY SUMMARY")
print("-" * 60)

print(f"Missing Values      : {final_missing:,}")
print(f"Duplicate Records   : {final_duplicates:,}")

print("\nMEMORY USAGE")
print("-" * 60)

print(f"Dataset Size (MB)   : {final_memory:.2f}")

print("=" * 60)

FINAL DATASET PROFILE
Rows                : 7,728,394
Columns             : 46

COLUMN TYPE SUMMARY
------------------------------------------------------------
Numeric Columns     : 13
Categorical Columns : 17
Datetime Columns    : 3
Boolean Columns     : 13

DATA QUALITY SUMMARY
------------------------------------------------------------
Missing Values      : 1,606,560
Duplicate Records   : 0

MEMORY USAGE
------------------------------------------------------------
Dataset Size (MB)   : 8367.27


## Step 13: AutoClean Final Report

In [17]:
# ==========================================================
# STEP 13: AUTOCLEAN FINAL REPORT
# ==========================================================
#
# Purpose:
# Compare dataset before and after cleaning.
#
# ==========================================================

# Calculate additional metrics
missing_filled = baseline_missing - final_missing

memory_saved = baseline_memory - final_memory

memory_reduction_pct = (
    memory_saved / baseline_memory
) * 100

print("=" * 62)
print("AUTOCLEAN FINAL REPORT")
print("=" * 62)

print(f"\nRows Before Cleaning      : {baseline_rows:,}")
print(f"Rows After Cleaning       : {len(df):,}")

print()

print(f"Missing Values Before     : {baseline_missing:,}")
print(f"Missing Values After      : {final_missing:,}")
print(f"Missing Values Filled     : {missing_filled:,}")

print()

print(f"Duplicates Before         : {baseline_duplicates:,}")
print(f"Duplicates After          : {final_duplicates:,}")

print()

print(f"Memory Usage Before (MB)  : {baseline_memory:.2f}")
print(f"Memory Usage After (MB)   : {final_memory:.2f}")
print(f"Memory Saved (MB)         : {memory_saved:.2f}")
print(f"Memory Reduction (%)      : {memory_reduction_pct:.2f}%")

print()

print("Dataset Status            : READY FOR ANALYSIS")

print("\n" + "=" * 62)

AUTOCLEAN FINAL REPORT

Rows Before Cleaning      : 7,728,394
Rows After Cleaning       : 7,728,394

Missing Values Before     : 12,840,498
Missing Values After      : 1,606,560
Missing Values Filled     : 11,233,938

Duplicates Before         : 0
Duplicates After          : 0

Memory Usage Before (MB)  : 9695.59
Memory Usage After (MB)   : 8367.27
Memory Saved (MB)         : 1328.32
Memory Reduction (%)      : 13.70%

Dataset Status            : READY FOR ANALYSIS



## Step 14: Export Clean Dataset

In [19]:
# ==========================================================
# STEP 14: EXPORT CLEAN DATASET
# ==========================================================
#
# Purpose:
# Save cleaned dataset to data/ folder
#
# ==========================================================

output_path = "../data/cleaned_dataset.csv"

df.to_csv(
    output_path,
    index=False
)

print("=" * 60)
print("DATASET EXPORT REPORT")
print("=" * 60)

print("✓ Clean dataset exported successfully")
print(f"Location : {output_path}")

print("=" * 60)

DATASET EXPORT REPORT
✓ Clean dataset exported successfully
Location : ../data/cleaned_dataset.csv


## Step 15: Export Cleaning Report

In [18]:
# ==========================================================
# STEP 15: EXPORT CLEANING REPORT
# ==========================================================
#
# Purpose:
# Create an Excel report summarizing cleaning results.
#
# ==========================================================

report_df = pd.DataFrame({
    "Metric": [
        "Rows Before",
        "Rows After",
        "Columns",
        "Missing Values Before",
        "Missing Values After",
        "Duplicates Before",
        "Duplicates After",
        "Memory Usage Before (MB)",
        "Memory Usage After (MB)"
    ],

    "Value": [
        baseline_rows,
        len(df),
        baseline_columns,
        baseline_missing,
        final_missing,
        baseline_duplicates,
        final_duplicates,
        round(baseline_memory, 2),
        round(final_memory, 2)
    ]
})

report_path = "../reports/autoclean_report.xlsx"

report_df.to_excel(
    report_path,
    index=False
)

print("=" * 60)
print("REPORT EXPORT SUCCESSFUL")
print("=" * 60)

print(f"Report Location : {report_path}")

print("=" * 60)

REPORT EXPORT SUCCESSFUL
Report Location : ../reports/autoclean_report.xlsx


In [16]:
df.isnull().sum().sort_values(ascending=False)

start_time               743166
end_time                 743166
weather_timestamp        120228
id                            0
railway                       0
precipitation(in)             0
weather_condition             0
amenity                       0
bump                          0
crossing                      0
give_way                      0
junction                      0
no_exit                       0
roundabout                    0
wind_direction                0
station                       0
stop                          0
traffic_calming               0
traffic_signal                0
turning_loop                  0
sunrise_sunset                0
civil_twilight                0
nautical_twilight             0
wind_speed(mph)               0
pressure(in)                  0
visibility(mi)                0
city                          0
severity                      0
start_lat                     0
start_lng                     0
end_lat                       0
end_lng 